# Step 5. Retrievers Benchmark

In [1]:
import os
import re
import pickle
import pandas as pd

from retriever.freq_retriever import FreqRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm.notebook import tqdm

In [2]:
SPLITS_PATH = './splits'
DATA_PATH = './data'

In [3]:
questions = pd.read_csv(os.path.join(DATA_PATH, 'questions.csv'), sep=';')
questions.head()

,question,chunk_id,section_name,chapter_name
0,"Что, по словам Терви, является его ""ружьем"" и ...",d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
1,Какой предмет использовал Терви для питания св...,d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
2,Какое телевизионное шоу любили смотреть Райдел...,d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
3,"Как зовут женщину, которая пришла к Райделлу в...",d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
4,"Какую функцию выполняли камеры, установленные ...",6851dc51ddcc613e44670dfc2c6ec6f3,Райделл вставил ключ...,"2. ""ГРОМИЛА"" В ДЕЛЕ"


In [4]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)
print(len(splits))

100


### Custom FreqRetriever Test

In [5]:
retriever = FreqRetriever.from_documents(
    docs=splits,
    k = 3,
    with_relevance = False
)

In [6]:
def retriever_test(query: str) -> list:
    result = retriever.invoke(query)
    if result:
        return result
    else:
        return "No relevant documents found"

In [7]:
query = """Что Райделл сообщил Шеветте о месте, куда учесал убийца, и как оно называется?"""
result = retriever_test(query)
if isinstance(result, list):
    print(result[0].metadata)

{'title': 'Уильям Гибсон. Виртуальный свет', 'chapter': '31. С ВОДИТЕЛЬСКОЙ СТОРОНЫ', 'section': 'Пуля прошила стенку...', 'id': 'f324e862a1854fddc3cd3034a74f2903', 'size': 3685, 'collection': 'Virtual_Light'}


In [8]:
def benchmark(retriever, verbose=True):
    num_correct = 0
    hit = 0
    rank = 0
    num_questions = 0
    for i in questions.index:
        
        num_questions += 1
        question = questions.loc[i, 'question']
        correct_chunk_id = questions.loc[i, 'chunk_id']
        result = retriever.invoke(question)

        if result and isinstance(result, list):
            chunks_ids = []
            for doc in result:
                chunks_ids.append(doc.metadata.get('id', ''))
            
            if verbose:
                print(f'Q: {question}')
                print(correct_chunk_id)
                print(chunks_ids)
            if any([idx == correct_chunk_id for idx in chunks_ids]):
                num_correct += 1
                rank += 1./(chunks_ids.index(correct_chunk_id) + 1)
                if chunks_ids.index(correct_chunk_id) == 0:
                    if verbose:
                        print(f'Found First\n')
                
                else:
                    if verbose:
                        print(f'Found\n')
            else:
                if verbose:
                    print(f'Not Found\n')
    
    print(f'Hit Rate: {round(num_correct/num_questions, 3)}')
    print(f'MRR: {round(rank/num_questions, 3)}')
    return None

In [9]:
%%time
benchmark(retriever=retriever, verbose=False)

Hit Rate: 0.859
MRR: 0.757
CPU times: total: 625 ms
Wall time: 614 ms


### Test LangChain BM25Retriever

In [10]:
retriever = BM25Retriever.from_documents(
    splits,
    k = 3,
)

In [11]:
print(retriever)

vectorizer=<rank_bm25.BM25Okapi object at 0x000001C61E205580> k=3


In [12]:
%%time
benchmark(retriever=retriever, verbose=False)

Hit Rate: 0.636
MRR: 0.556
CPU times: total: 188 ms
Wall time: 188 ms


### InMemory vector retriever test

In [13]:
embedding = HuggingFaceEmbeddings(model_name="deepvk/USER-bge-m3", model_kwargs={"device": "cpu"})

In [14]:
vector_store = InMemoryVectorStore(embedding)

In [15]:
%%time
for s in tqdm(splits):
    split_id = s.metadata.get('id')
    vector_store.add_documents(documents=[s], ids=[split_id])

  0%|          | 0/100 [00:00<?, ?it/s]

CPU times: total: 39min 23s
Wall time: 6min 44s


In [16]:
retriever = vector_store.as_retriever(k=3)
print(retriever)

tags=['InMemoryVectorStore', 'HuggingFaceEmbeddings'] vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x000001C62CD67EC0> search_kwargs={}


In [17]:
%%time
benchmark(retriever=retriever, verbose=False)

Hit Rate: 0.741
MRR: 0.589
CPU times: total: 3min 55s
Wall time: 39.8 s


### Qdrant vector store test

In [ ]:
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

In [ ]:
QDRANT_HOST = "127.0.0.1"
QDRANT_PORT = 6333
QDRANT_EMBEDDING_SIZE = 1024
QDRANT_DISTANCE = Distance.COSINE

In [ ]:
qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [ ]:
qdrant_client.create_collection(
    collection_name='virtual_light',
    vectors_config=VectorParams(size=QDRANT_EMBEDDING_SIZE, distance=QDRANT_DISTANCE),
)

In [ ]:
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name='virtual_light',
    distance=QDRANT_DISTANCE,
    embedding=embedding,
)

In [ ]:
%%time
for s in tqdm(splits):
    split_id = s.metadata.get('id')
    vector_store.add_documents(documents=[s], ids=[split_id])    

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(retriever)

In [ ]:
result = retriever_test(query)
if isinstance(result, list):
    print(result[0].metadata)
    print(result[1].metadata)
    print(result[2].metadata)

In [ ]:
%%time
benchmark(retriever=retriever, verbose=False)